# NB03 — Harmonise, cohort flow, Table 1, splits

**Purpose.** Apply identical cohort criteria to both databases, pivot the raw aggregates into the analysis feature set, build the CONSORT flow and Table 1 (with standardised mean differences), and define the pre-specified splits (MIMIC development vs temporal test; eICU 2014 update vs 2015 evaluation).

**Research question.** RQ0 — is the pipeline clean and harmonised?

| | |
|---|---|
| **Inputs** | `data/raw/*.parquet` from NB01–NB02 |
| **Outputs** | `data/analysis/mimic_analysis.parquet`, `eicu_analysis.parquet`; Table 1; flow |
| **Free-Colab runtime** | ~5 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB03')

In [ ]:
def raw(name):
    return pd.read_parquet(os.path.join(tu.DATA_RAW, name + '.parquet'))

def to_num(df, cols):
    for c in cols:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype(float)
    return df

def pivot_features(vit, lab, gcs):
    """Long (stay, var) aggregates -> one row per stay with harmonised feature names."""
    vit = to_num(vit.copy(), ['stay_id', 'vmin', 'vmax', 'vmean', 'n_hours', 'rmin', 'rmax', 'rmean', 'n_raw'])
    lab = to_num(lab.copy(), ['stay_id', 'vmin', 'vmax', 'vmean', 'n_raw'])
    gcs = to_num(gcs.copy(), ['stay_id', 'gcs_min', 'gcs_n'])
    wv = vit.pivot_table(index='stay_id', columns='var',
                         values=['vmin', 'vmax', 'vmean', 'rmin', 'rmax', 'rmean', 'n_raw', 'n_hours'],
                         aggfunc='first')
    wl = lab.pivot_table(index='stay_id', columns='var', values=['vmin', 'vmax', 'n_raw'], aggfunc='first')
    idx = wv.index.union(wl.index).union(pd.Index(gcs['stay_id']))
    cols = {}
    get = lambda w, a, v: w[(a, v)].reindex(idx) if (a, v) in w.columns else pd.Series(np.nan, index=idx)
    for v in tu.VITALS:
        for s in ('min', 'max', 'mean'):
            cols[f'{v}_{s}'] = get(wv, 'v' + s, v)
            cols[f'raw_{v}_{s}'] = get(wv, 'r' + s, v)
        cols[f'cnt_{v}'] = get(wv, 'n_raw', v).fillna(0)
        cols[f'nh_{v}'] = get(wv, 'n_hours', v).fillna(0)
    for l in tu.LABS:
        cols[f'{l}_min'] = get(wl, 'vmin', l)
        cols[f'{l}_max'] = get(wl, 'vmax', l)
        cols[f'cnt_{l}'] = get(wl, 'n_raw', l).fillna(0)
    g = gcs.set_index('stay_id').reindex(idx)
    cols['gcs_min'] = g['gcs_min']
    cols['cnt_gcs'] = g['gcs_n'].fillna(0)
    out = pd.DataFrame(cols, index=idx)
    out.index.name = 'stay_id'
    return out.reset_index()

def race_group(s):
    s = s.fillna('').str.upper()
    return np.select([s.str.contains('WHITE|CAUCASIAN'), s.str.contains('BLACK|AFRICAN'),
                      s.str.contains('HISPANIC|LATINO'), s.str.contains('ASIAN')],
                     ['White', 'Black', 'Hispanic', 'Asian'], 'Other/Unknown')

### MIMIC-IV cohort

In [ ]:
mc = to_num(raw('mimic_cohort'), ['stay_id', 'subject_id', 'rn', 'icu_los_h', 'age', 'mortality', 'hours_to_death'])
mfeat = pivot_features(raw('mimic_vitals'), raw('mimic_labs'), raw('mimic_gcs'))

flow_m = []
def step(label, d):
    flow_m.append(dict(dataset='MIMIC-IV', step=label, n=len(d), deaths=int(d['mortality'].sum()),
                       mortality_pct=round(100 * d['mortality'].mean(), 2)))
    return d
d = step('All ICU stays', mc)
d = step('First ICU stay per patient', d[d['rn'] == 1])
d = step('Age >= 18', d[d['age'] >= 18])
d = step('ICU stay >= 24 h', d[d['icu_los_h'] >= 24])
d = step('Alive at 24 h', d[~(d['hours_to_death'].notna() & (d['hours_to_death'] < 24))])
d = d.merge(mfeat, on='stay_id', how='left')
d['cnt_hr'] = d['cnt_hr'].fillna(0)
d = step('>= 1 heart-rate value in first 24 h', d[d['cnt_hr'] > 0])

mimic = d.copy()
mimic['sex_male'] = (mimic['gender'] == 'M').astype(float)
mimic['race_group'] = race_group(mimic['race'])
mimic['age'] = mimic['age'].clip(upper=90)
mimic['period'] = np.where(mimic['anchor_year_group'].isin(tu.MIMIC_DEV_GROUPS), 'dev', 'temporal_test')
mimic['patient_id'] = mimic['subject_id']
print(pd.DataFrame(flow_m).to_string(index=False))
print(mimic.groupby('anchor_year_group')['mortality'].agg(['size', 'mean']))

### eICU cohort

In [ ]:
ec = to_num(raw('eicu_cohort'), ['stay_id', 'hospital_id', 'rn', 'year', 'age', 'unitdischargeoffset',
                                 'hospitaldischargeoffset'])
ec['mortality'] = (ec['hospitaldischargestatus'] == 'Expired').astype(float)
efeat = pivot_features(raw('eicu_vitals'), raw('eicu_labs'), raw('eicu_gcs'))
hosp = raw('eicu_hospital'); hosp['hospital_id'] = pd.to_numeric(hosp['hospital_id'])
ap = to_num(raw('eicu_apache'), ['stay_id', 'apache_pred', 'apache_score'])
ap.loc[ap['apache_pred'] < 0, 'apache_pred'] = np.nan          # -1 = not computed
ap = ap.drop_duplicates('stay_id')

flow_e = []
def step_e(label, d):
    flow_e.append(dict(dataset='eICU-CRD', step=label, n=len(d), deaths=int(d['mortality'].sum()),
                       mortality_pct=round(100 * d['mortality'].mean(), 2)))
    return d
d = step_e('All ICU stays', ec)
d = step_e('Known hospital discharge status', d[d['hospitaldischargestatus'].isin(['Alive', 'Expired'])])
d = step_e('First ICU stay per patient', d[d['rn'] == 1])
d = step_e('Age >= 18', d[d['age'] >= 18])
d = step_e('ICU stay >= 24 h', d[d['unitdischargeoffset'] >= 1440])
d = step_e('Alive at 24 h', d[~((d['mortality'] == 1) & (d['hospitaldischargeoffset'] < 1440))])
d = d.merge(efeat, on='stay_id', how='left')
d['cnt_hr'] = d['cnt_hr'].fillna(0)
d = step_e('>= 1 heart-rate value in first 24 h', d[d['cnt_hr'] > 0])

eicu = d.merge(hosp, on='hospital_id', how='left').merge(ap, on='stay_id', how='left')
g = eicu['gender'].fillna('')
eicu['sex_male'] = np.where(g == 'Male', 1.0, np.where(g == 'Female', 0.0, np.nan))
eicu['race_group'] = race_group(eicu['ethnicity'])
eicu['age'] = eicu['age'].clip(upper=90)
eicu['split'] = np.select([eicu['year'] == tu.EICU_UPDATE_YEAR, eicu['year'] == tu.EICU_EVAL_YEAR],
                          ['update', 'eval'], 'other')
# sensitivity split: random 50/50 within each hospital (stratified by outcome)
rng = np.random.default_rng(tu.SEED)
eicu['random_split'] = 'eval'
for (h, yv), idx in eicu.groupby(['hospital_id', 'mortality']).groups.items():
    idx = np.array(list(idx)); rng.shuffle(idx)
    eicu.loc[idx[: len(idx) // 2], 'random_split'] = 'update'
eicu['patient_id'] = eicu['uniquepid']
print(pd.DataFrame(flow_e).to_string(index=False))
print('\nHospitals:', eicu['hospital_id'].nunique(), '| split counts:', eicu['split'].value_counts().to_dict())
print('APACHE IVa available:', round(100 * eicu['apache_pred'].notna().mean(), 1), '%')

### Unit and plausibility check (harmonisation audit)

In [ ]:
def smd(a, b):
    a, b = a.dropna(), b.dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    s = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - b.mean()) / s if s > 0 else np.nan

audit_cols = tu.FEATURES_V1 + tu.COUNT_FEATS + [c for c in tu.V2_VITAL]
rows = []
for c in audit_cols:
    a, b = mimic[c], eicu[c]
    rows.append(dict(feature=c, mimic_median=a.median(), eicu_median=b.median(),
                     mimic_missing_pct=100 * a.isna().mean(), eicu_missing_pct=100 * b.isna().mean(),
                     smd=smd(a, b)))
audit = pd.DataFrame(rows)
audit['abs_smd'] = audit['smd'].abs()
tu.save_table(audit.round(3).sort_values('abs_smd', ascending=False), 'feature_audit_smd')
print('\nLargest shifts among PRIMARY (V1) features:')
print(audit[audit['feature'].isin(tu.FEATURES_V1)].sort_values('abs_smd', ascending=False).head(10)
      [['feature', 'mimic_median', 'eicu_median', 'smd']].round(2).to_string(index=False))

### Table 1

In [ ]:
def describe(d, label):
    q = lambda s: f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"
    r = {'Cohort': label, 'N': f"{len(d):,}",
         'Hospitals': d['hospital_id'].nunique() if 'hospital_id' in d else 1,
         'Age, median (IQR)': q(d['age']), 'Male, %': f"{100 * d['sex_male'].mean():.1f}",
         'In-hospital mortality, %': f"{100 * d['mortality'].mean():.2f}",
         'GCS min, median (IQR)': q(d['gcs_min']),
         'Lactate measured, %': f"{100 * d['lactate_max'].notna().mean():.1f}",
         'MBP mean, median (IQR)': q(d['mbp_mean']), 'Creatinine max, median (IQR)': q(d['creat_max'])}
    for rg in ['White', 'Black', 'Hispanic', 'Asian', 'Other/Unknown']:
        r[f'Race: {rg}, %'] = f"{100 * (d['race_group'] == rg).mean():.1f}"
    if 'apache_pred' in d:
        r['APACHE IVa predicted mortality, mean %'] = f"{100 * d['apache_pred'].mean():.2f}"
    return r

t1 = pd.DataFrame([describe(mimic[mimic.period == 'dev'], 'MIMIC-IV development (2008-2016)'),
                   describe(mimic[mimic.period == 'temporal_test'], 'MIMIC-IV temporal test (2017-2022)'),
                   describe(eicu[eicu.split == 'update'], 'eICU 2014 (update)'),
                   describe(eicu[eicu.split == 'eval'], 'eICU 2015 (evaluation)')]).set_index('Cohort').T
tu.save_table(t1, 'table1', index=True)

### Hospital eligibility for per-hospital analyses

In [ ]:
hy = eicu.groupby(['hospital_id', 'split']).agg(n=('mortality', 'size'), deaths=('mortality', 'sum')).unstack(fill_value=0)
hy.columns = [f'{a}_{b}' for a, b in hy.columns]
for c in ['n_update', 'n_eval', 'deaths_eval', 'deaths_update']:
    if c not in hy: hy[c] = 0
hy['eligible_heterogeneity_all_years'] = eicu.groupby('hospital_id').apply(
    lambda g: (len(g) >= tu.MIN_HOSP_N) and (g['mortality'].sum() >= tu.MIN_HOSP_EVENTS))
hy['eligible_temporal'] = (hy['n_update'] >= tu.MIN_UPDATE_N) & (hy['n_eval'] >= tu.MIN_HOSP_N) & \
                          (hy['deaths_eval'] >= tu.MIN_HOSP_EVENTS)
tu.save_table(hy.reset_index(), 'hospital_year_counts', show=False)
elig = dict(hospitals_total=int(len(hy)),
            eligible_heterogeneity=int(hy['eligible_heterogeneity_all_years'].sum()),
            eligible_temporal=int(hy['eligible_temporal'].sum()),
            patients_in_eligible_temporal_eval=int(hy.loc[hy['eligible_temporal'], 'n_eval'].sum()))
print(elig)

In [ ]:
keep_common = ['stay_id', 'patient_id', 'mortality', 'age', 'sex_male', 'race_group'] + \
              [c for c in tu.FEATURES_V1 if c not in tu.DEMO] + tu.V2_VITAL + tu.COUNT_FEATS + \
              [f'nh_{v}' for v in tu.VITALS]
mimic_out = mimic[keep_common + ['anchor_year_group', 'period', 'first_careunit']].copy()
eicu_out = eicu[keep_common + ['hospital_id', 'year', 'split', 'random_split', 'apache_pred', 'apache_score',
                               'numbedscategory', 'teachingstatus', 'region', 'unittype']].copy()
for df_ in (mimic_out, eicu_out):
    df_['patient_id'] = df_['patient_id'].astype(str)
    df_['race_group'] = df_['race_group'].astype(str)
eicu_out['teachingstatus'] = eicu_out['teachingstatus'].astype(str)
mimic_out.to_parquet(os.path.join(tu.DATA_AN, 'mimic_analysis.parquet'), index=False)
eicu_out.to_parquet(os.path.join(tu.DATA_AN, 'eicu_analysis.parquet'), index=False)
flow = pd.DataFrame(flow_m + flow_e)
tu.save_table(flow, 'cohort_flow', show=False)
print('Saved analysis files:', mimic_out.shape, eicu_out.shape)

In [ ]:
tu.finish_notebook({
    'flow': flow.to_dict('records'),
    'mimic_periods': mimic_out['period'].value_counts().to_dict(),
    'mimic_mortality_by_period': mimic_out.groupby('period')['mortality'].mean().round(4).to_dict(),
    'eicu_splits': eicu_out['split'].value_counts().to_dict(),
    'eicu_mortality_by_split': eicu_out.groupby('split')['mortality'].mean().round(4).to_dict(),
    'apache_available_pct': round(100 * eicu_out['apache_pred'].notna().mean(), 1),
    'hospital_eligibility': elig,
    'top_shifted_primary_features': audit[audit['feature'].isin(tu.FEATURES_V1)]
        .sort_values('abs_smd', ascending=False).head(8)[['feature', 'smd']].round(3).to_dict('records'),
    'count_feature_smd': audit[audit['feature'].str.startswith('cnt_')][['feature', 'smd']].round(3).to_dict('records'),
})